[Project Stone]
- 돌 분류 프로젝트

In [36]:
import sys
import torch
import os

print("--- Environment Check ---")
print(f"Python Executable: {sys.executable}") # 현재 사용 중인 파이썬 실행 파일 경로
print(f"Python Version: {sys.version}")      # 현재 사용 중인 파이썬 버전
print(f"PyTorch Version: {torch.__version__}") # 현재 로드된 PyTorch 버전
print(f"PyTorch CUDA Build: {torch.version.cuda if hasattr(torch.version, 'cuda') else 'N/A'}") # PyTorch가 빌드된 CUDA 버전
print(f"CUDA Available: {torch.cuda.is_available()}") # CUDA 사용 가능 여부
if torch.cuda.is_available():
    print(f"CUDA Version (Runtime): {torch.version.cuda}") # PyTorch가 인식하는 런타임 CUDA 버전
    print(f"Device Name: {torch.cuda.get_device_name(0)}") # GPU 이름
    print(f"Device Compute Capability: {torch.cuda.get_device_capability(0)}") # Compute Capability 확인
print("-" * 25)

--- Environment Check ---
Python Executable: c:\ProgramData\anaconda3\envs\DL_P310\python.exe
Python Version: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
PyTorch Version: 2.4.0
PyTorch CUDA Build: 12.4
CUDA Available: True
CUDA Version (Runtime): 12.4
Device Name: NVIDIA A100 80GB PCIe
Device Compute Capability: (8, 0)
-------------------------


In [37]:
import pandas as pd
import numpy as np

In [38]:
numlist = os.listdir('./_data/open/train')
trainDIR = './_data/open/train'
sum = 0

for i in numlist:
    a = (len(os.listdir('./_data/open/train/'+i)))
    print(i, a)
    sum += a
print(sum)

Andesite 43802
Basalt 26810
Etc 15935
Gneiss 73914
Granite 92923
Mud_Sandstone 89467
Weathered_Rock 37169
380020


In [39]:
numlist = os.listdir('./_data/open/test')
testDIR = './_data/open/test'
len(numlist)

95006

In [40]:
## 작업순서
## 데이테 셋 만들기 
## 데이터 로더 만들기
## 폴더별로 되있으니까 imgeafolder사용하면 될듯?


In [41]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from torch.utils.data import Dataset, DataLoader  # Pytorch의 데이터셋 관련
from torchvision import transforms  # 전처리모듈
from torchvision.datasets import ImageFolder

from PIL import Image


In [42]:
# TRANSFORM = transforms.Compose([
#         transforms.Resize((380, 380)),  # B7 전용 입력 크기
#         transforms.ToTensor(),
#         transforms.Normalize(
#             mean=[0.485, 0.456, 0.406],  # ImageNet 평균값
#             std=[0.229, 0.224, 0.225]    # ImageNet 표준편차
#         )
# ])

TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(380, scale=(0.8, 1.0)),  # 확대·축소 후 crop
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

TRANSFORM_MINOR = transforms.Compose([
    transforms.RandomResizedCrop(380, scale=(0.8, 1.0)),  # 확대·축소 후 crop
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [43]:
transform_dict = {
 'Andesite': TRANSFORM,
 'Basalt': TRANSFORM,
 'Gneiss': TRANSFORM,
 'Granite': TRANSFORM,
 'Mud_Sandstone': TRANSFORM,
 'Weathered_Rock': TRANSFORM,
 'Etc': TRANSFORM
}
# transform_dict = {
#  'Andesite': TRANSFORM_MINOR,
#  'Basalt': TRANSFORM_MINOR,
#  'Gneiss': TRANSFORM,
#  'Granite': TRANSFORM,
#  'Mud_Sandstone': TRANSFORM,
#  'Weathered_Rock': TRANSFORM_MINOR,
#  'Etc': TRANSFORM_MINOR
# }

In [44]:
classes = ['Andesite', 'Basalt', 'Gneiss', 'Granite', 'Mud_Sandstone', 'Weathered_Rock']
classes2idx = {x:idx+1 for idx,x in enumerate(classes)}
classes2idx['Etc']= 0
classes2idx

{'Andesite': 1,
 'Basalt': 2,
 'Gneiss': 3,
 'Granite': 4,
 'Mud_Sandstone': 5,
 'Weathered_Rock': 6,
 'Etc': 0}

In [45]:
class CustomImageFolder(ImageFolder):
    def __init__(self, root, transform_dict, classes2idx):
        self.transform_dict = transform_dict              # 클래스별 transform 저장
        self.classes2idx = classes2idx
        self.class_to_idx = classes2idx
        self.classes = list(classes2idx.keys())
        super().__init__(root, transform=None)            # transform은 직접 처리할 것이므로 None

    def find_classes(self, directory):
        # 고정된 클래스 사용
        return self.classes, self.classes2idx

    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = Image.open(path).convert("RGB")

        # 클래스 인덱스를 다시 클래스명으로 매핑
        class_name = self.classes[target]
        transform = self.transform_dict.get(class_name)

        if transform:
            sample = transform(sample)

        return sample, target

In [46]:
trainDS = CustomImageFolder(trainDIR, transform_dict, classes2idx=classes2idx)

In [47]:
print('train', len(trainDS))
print(trainDS.classes)
trainDS.class_to_idx
idx2class = {values:key for key, values in classes2idx.items()}
idx2class

train 380020
['Andesite', 'Basalt', 'Gneiss', 'Granite', 'Mud_Sandstone', 'Weathered_Rock', 'Etc']


{1: 'Andesite',
 2: 'Basalt',
 3: 'Gneiss',
 4: 'Granite',
 5: 'Mud_Sandstone',
 6: 'Weathered_Rock',
 0: 'Etc'}

In [48]:
trainDS[111100][0].shape

torch.Size([3, 380, 380])

In [49]:
cnum = 0
for (a, b) in trainDS:
    cnum += 1
    print(a[0].shape, idx2class[b])
    if cnum == 10: break
    

torch.Size([380, 380]) Andesite
torch.Size([380, 380]) Andesite
torch.Size([380, 380]) Andesite
torch.Size([380, 380]) Andesite
torch.Size([380, 380]) Andesite
torch.Size([380, 380]) Andesite
torch.Size([380, 380]) Andesite
torch.Size([380, 380]) Andesite
torch.Size([380, 380]) Andesite
torch.Size([380, 380]) Andesite


In [50]:
from torch.utils.data import random_split

# 전체 데이터 수
total_size = len(trainDS)
train_size = int(0.8 * total_size)
valid_size = total_size - train_size

# 무작위로 train/valid 분리
trainDS, validDS = random_split(trainDS, [train_size, valid_size])

In [51]:
print(len(trainDS))
print(len(validDS))

304016
76004


In [52]:
imgTS, label = trainDS[0]   ## __getitem__(index)
print(imgTS.shape, label)


torch.Size([3, 380, 380]) 6


In [53]:
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches

In [54]:
# plt.imshow(imgTS.permute(1,2,0))
# plt.title(idx2class[label])
# plt.show()

In [55]:
def collator(batch):
    images, labels = zip(*batch)  # 튜플 of Tensors
    images = torch.stack(images)  # → Tensor of shape [B, C, H, W]
    labels = torch.tensor(labels) # → Tensor of shape [B]
    return images, labels

BATCH_SIZE = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 0.0001
DEVICE

'cuda'

In [56]:
trainDL = DataLoader(
    trainDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)
validDL = DataLoader(
    validDS, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, collate_fn=collator
)

In [57]:
for a, b in trainDL:
    print((type(a)),(type(b)))
    break

<class 'torch.Tensor'> <class 'torch.Tensor'>


In [58]:
from torchvision import models
from torchvision import ops
from torchvision.models.detection import rpn

num_classes = len(classes2idx)
num_classes

7

In [59]:
teacher = models.resnet101(pretrained=False)
teacher.fc = nn.Linear(teacher.fc.in_features, num_classes)
teacher.load_state_dict(torch.load('./_model/4_vl0.29.pth'))  # 학습된 모델 불러오기
teacher.eval()
teacher.to(DEVICE)

C:\Users\K\AppData\Local\Temp\ipykernel_21860\2436901433.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher.load_state_dict(torch.load('./_model/4_vl0.29.pth'))  # 

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [60]:
model = models.efficientnet_b4(pretrained=True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model.load_state_dict(torch.load('./_model/EN/ENBT_3_vl0.3962.pth'))  # 학습된 모델 불러오기
model.to(DEVICE)

C:\Users\K\AppData\Local\Temp\ipykernel_21860\805704054.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('./_model/EN/ENBT_3_vl0.3962.pth

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
            (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActiv

In [61]:
def distillation_loss(y_student, y_teacher, y_true, T=4, alpha=0.7):
    # Soft target loss
    soft_loss = F.kl_div(
        F.log_softmax(y_student / T, dim=1),
        F.softmax(y_teacher / T, dim=1),
        reduction='batchmean'
    ) * (T * T)

    # Hard target loss
    hard_loss = F.cross_entropy(y_student, y_true)

    # α는 hard target의 비중
    return alpha * hard_loss + (1 - alpha) * soft_loss

# def distillation_loss(y_student, y_teacher, y_true, T=4, alpha=0.7):
#     # Soft target (teacher 예측을 모방)
#     soft_loss = F.kl_div(
#         F.log_softmax(y_student / T, dim=1),
#         F.softmax(y_teacher / T, dim=1),
#         reduction='batchmean'
#     ) * (T * T)

#     # Hard target (정답과의 비교)
#     hard_loss = F.cross_entropy(y_student, y_true)

#     return alpha * soft_loss + (1 - alpha) * hard_loss

In [34]:
from torch import optim
from tqdm import tqdm
# ✅ 손실함수, 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


In [35]:
torch.__version__

'2.4.0'

In [62]:
# %pip install psutil
import psutil
import subprocess
from tqdm import tqdm

def get_gpu_usage():
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,nounits,noheader"]
        )
        result = result.decode("utf-8").strip().split("\n")[0]
        gpu_util, mem_used, mem_total = map(int, result.split(", "))
        return gpu_util, mem_used, mem_total
    except Exception:
        return None, None, None

In [63]:
import time

In [ ]:
now = time.localtime()
ct = time.strftime("%H%M",now)
ct

'25.05.05 09:40:58'

In [ ]:

EPOCH = 5
best_val_loss = float('inf')
patience = 3
patience_counter = 0

# 에포크별 '평균' 지표를 저장할 리스트
train_metrics = []
val_metrics = []

# psutil.cpu_percent() 초기 호출 (첫 호출 시 의미 없는 값 반환 방지)
psutil.cpu_percent(interval=None)
time.sleep(0.1) # CPU 사용률 측정을 위한 짧은 대기

for epoch in range(EPOCH):
    model.train()
    epoch_train_loss_sum = 0.0
    epoch_train_correct = 0
    epoch_train_total = 0
    epoch_train_cpu_sum = 0.0
    epoch_train_gpu_sum = 0.0
    train_batches = 0

    train_loop = tqdm(trainDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Training", leave=False)
    for images, labels in train_loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        with torch.no_grad():
            teacher_outputs = teacher(images)

        student_outputs = model(images)
        loss = distillation_loss(student_outputs, teacher_outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_train_loss_sum += loss.item()
        _, predicted = student_outputs.max(1)
        epoch_train_correct += predicted.eq(labels).sum().item()
        epoch_train_total += labels.size(0)
        train_batches += 1

        cpu_usage = psutil.cpu_percent(interval=None)
        gpu_util, _, _ = get_gpu_usage()

        epoch_train_cpu_sum += cpu_usage
        epoch_train_gpu_sum += gpu_util if gpu_util is not None else 0

        current_cumulative_acc = 100. * epoch_train_correct / epoch_train_total
        train_loop.set_postfix(loss=loss.item(),
                               acc=f"{current_cumulative_acc:.2f}%",
                               cpu=f"{cpu_usage:.1f}%",
                               gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")


    # --- 에포크 평균 계산 ---
    # train_batches가 0인 경우 방지
    if train_batches > 0:
        avg_train_loss = epoch_train_loss_sum / train_batches
        avg_train_acc = 100. * epoch_train_correct / epoch_train_total
        avg_train_cpu = epoch_train_cpu_sum / train_batches
        avg_train_gpu = epoch_train_gpu_sum / train_batches
    else:
        avg_train_loss, avg_train_acc, avg_train_cpu, avg_train_gpu = 0, 0, 0, 0

    # --- 에포크 평균 지표 저장 ---
    train_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_train_loss,
        "accuracy": avg_train_acc,
        "avg_cpu_percent": avg_train_cpu,
        "avg_gpu_percent": avg_train_gpu
    })

    # ----------- Validation -----------
    model.eval()
    # --- 에포크 누적 변수 초기화 ---
    epoch_val_loss_sum = 0.0
    epoch_val_correct = 0
    epoch_val_total = 0
    epoch_val_cpu_sum = 0.0
    epoch_val_gpu_sum = 0.0
    val_batches = 0 # 실제 처리된 배치 수 카운트

    val_loop = tqdm(validDL, desc=f"[Epoch {epoch+1}/{EPOCH}] Validation", leave=False)
    with torch.no_grad():
        for images, labels in val_loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            # --- 배치 결과 누적 ---
            epoch_val_loss_sum += loss.item()
            _, predicted = outputs.max(1)
            epoch_val_correct += predicted.eq(labels).sum().item()
            epoch_val_total += labels.size(0)
            val_batches += 1

            # --- 시스템 지표 측정 및 누적 ---
            cpu_usage = psutil.cpu_percent(interval=None)
            gpu_util, _, _ = get_gpu_usage()

            epoch_val_cpu_sum += cpu_usage
            epoch_val_gpu_sum += gpu_util if gpu_util is not None else 0

            # --- tqdm 진행률 표시줄 업데이트 (현재 '배치' 손실과 '누적' 정확도 표시) ---
            current_cumulative_acc = 100. * epoch_val_correct / epoch_val_total
            val_loop.set_postfix(loss=loss.item(), # 현재 배치 손실
                                 acc=f"{current_cumulative_acc:.2f}%", # 현재까지 누적 정확도
                                 cpu=f"{cpu_usage:.1f}%",
                                 gpu=f"{gpu_util if gpu_util is not None else 0:.1f}%")

    # --- 에포크 평균 계산 ---
    # val_batches가 0인 경우 방지
    if val_batches > 0:
        avg_val_loss = epoch_val_loss_sum / val_batches
        avg_val_acc = 100. * epoch_val_correct / epoch_val_total
        avg_val_cpu = epoch_val_cpu_sum / val_batches
        avg_val_gpu = epoch_val_gpu_sum / val_batches
    else:
        avg_val_loss, avg_val_acc, avg_val_cpu, avg_val_gpu = 0, 0, 0, 0

    # --- 에포크 평균 지표 저장 ---
    val_metrics.append({
        "epoch": epoch + 1,
        "loss": avg_val_loss,
        "accuracy": avg_val_acc,
        "avg_cpu_percent": avg_val_cpu,
        "avg_gpu_percent": avg_val_gpu
    })

    # --- 결과 출력 및 모델 저장 로직 (Early Stopping 포함 가능) ---
    print(f"\n[Epoch {epoch+1}/{EPOCH}]")
    print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.2f}% | Avg CPU: {avg_train_cpu:.1f}%, Avg GPU: {avg_train_gpu:.1f}%")
    print(f"  Valid Loss: {avg_val_loss:.4f}, Valid Acc: {avg_val_acc:.2f}% | Avg CPU: {avg_val_cpu:.1f}%, Avg GPU: {avg_val_gpu:.1f}%")

    #-- 현재 시각 
    now = time.localtime()
    ct = time.strftime("%H%M",now)
    
    
    # Early Stopping 및 모델 저장 (평균 검증 손실 사용)
    # 원본 코드의 저장 조건 유지 (val_loss_avg < 2 부분은 필요에 따라 제거/수정)
    if avg_val_loss < best_val_loss : # and avg_val_loss < 2:
        print(f"  Validation loss improved ({best_val_loss:.4f} --> {avg_val_loss:.4f}). Saving ENBT_{ct}_{epoch+1}_vl{avg_val_loss:.4f}.pth")
        best_val_loss = avg_val_loss
        # 모델 저장 경로 및 이름 확인 필요 (_model 폴더가 있어야 함)
        try:
            torch.save(model.state_dict(), f"./_model/EN/ENBT_{ct}_{epoch+1}_vl{avg_val_loss:.4f}.pth")
        except FileNotFoundError:
            print("  Warning: './_model/EN/' directory not found. Skipping model saving.")
        patience_counter = 0 # 개선되었으므로 카운터 초기화
    else:
        patience_counter += 1
        print(f"  Validation loss did not improve from {best_val_loss:.4f}. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"  Early stopping triggered after {epoch + 1} epochs.")
            break # 학습 중단
    
    train_df = pd.DataFrame(train_metrics)
    val_df = pd.DataFrame(val_metrics)
    train_df.to_csv("./_data/metrics_EN/train_metrics_ENBT.csv", index=False)
    val_df.to_csv("./_data/metrics_EN/val_metrics_ENBT.csv", index=False)
    
# 학습 완료 후 저장된 지표 확인 (예시)
print("\n--- Training Metrics (Epoch Averages) ---")
for i, metrics in enumerate(train_metrics):
    print(f"Epoch {metrics['epoch']}: {metrics}")

print("\n--- Validation Metrics (Epoch Averages) ---")
for i, metrics in enumerate(val_metrics):
    print(f"Epoch {metrics['epoch']}: {metrics}")


# -------------------------------
# train_metrics 와 val_metrics 리스트에는 이제 각 에포크의 평균 지표가
# 딕셔너리 형태로 저장되어 있습니다.
# 예: train_metrics[0] -> 첫 번째 에포크의 평균 {loss, accuracy, avg_cpu_percent, avg_gpu_percent}
# -------------------------------



[Epoch 1/5] Training:   0%|          | 0/30401 [00:00<?, ?it/s]

In [ ]:
os.listdir('../EXAM_DL/SOURCE/')

['check.ipynb', 'version_check.ipynb', 'version_check.py']

In [ ]:
# # 이제 이 데이터를 pandas DataFrame으로 변환하여 CSV로 저장할 수 있습니다.
# # 예시:
# train_df = pd.DataFrame(train_metrics)
# val_df = pd.DataFrame(val_metrics)
# train_df.to_csv("./_data/metrics_EN/train_metrics_ENBT.csv", index=False)
# val_df.to_csv("./_data/metrics_EN/val_metrics_ENBT.csv", index=False)

NameError: name 'train_metrics' is not defined

In [5]:
# import torch
# print(torch.__version__)             # ex) 2.4.1
# print(torch.version.cuda)           # ex) 12.4
# print(torch.cuda.get_device_name(0)) # ex) NVIDIA RTX 5090
# print(torch.cuda.is_available()) 

In [64]:
sampleDF = pd.DataFrame(pd.read_csv('./_data/open/sample_submission.csv'))
print(sampleDF.head())
testDF = pd.DataFrame(pd.read_csv('./_data/open/test.csv'))
print(testDF.head())

           ID rock_type
0  TEST_00000       Etc
1  TEST_00001       Etc
2  TEST_00002       Etc
3  TEST_00003       Etc
4  TEST_00004       Etc
           ID               img_path
0  TEST_00000  ./test/TEST_00000.jpg
1  TEST_00001  ./test/TEST_00001.jpg
2  TEST_00002  ./test/TEST_00002.jpg
3  TEST_00003  ./test/TEST_00003.jpg
4  TEST_00004  ./test/TEST_00004.jpg


In [74]:
class TestImageDataset(Dataset):
    def __init__(self, csv_df, image_root, transform=None):
        self.df = csv_df
        self.image_root = image_root  # 예: './_data/open/test/'
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0]
        img_path = self.df.iloc[idx, 1]
        img_path = os.path.join(self.image_root, img_path)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name


In [75]:
TRANSFORM_T = transforms.Compose([
        transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



In [76]:
testDS = TestImageDataset(testDF, image_root="./_data/open/", transform=TRANSFORM_T)
testDL = DataLoader(testDS, batch_size=32, shuffle=False)

In [77]:
model.eval()
predictions = []

with torch.no_grad():
    count = 0
    for images, filenames in testDL:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        
        for fname, pred in zip(filenames, preds):
            predictions.append((fname, idx2class[pred.item()]))
        count += 1
        # if count ==3: break

In [78]:
sampleDF.head()

,ID,rock_type
0,TEST_00000,Mud_Sandstone
1,TEST_00001,Etc
2,TEST_00002,Mud_Sandstone
3,TEST_00003,Granite
4,TEST_00004,Granite


In [79]:
# for i in range(sampleDF.shape[0]):
for i in range(95006):
    if sampleDF.loc[i,'ID'] == predictions[i][0]:
        sampleDF.loc[i,'rock_type'] = predictions[i][1]
    else:
        continue
sampleDF.head(20)

,ID,rock_type
0,TEST_00000,Mud_Sandstone
1,TEST_00001,Etc
2,TEST_00002,Mud_Sandstone
3,TEST_00003,Granite
4,TEST_00004,Granite
5,TEST_00005,Granite
6,TEST_00006,Granite
7,TEST_00007,Granite
8,TEST_00008,Granite
9,TEST_00009,Granite


In [80]:
now = time.localtime()
ct = time.strftime("%H%M",now)
sampleDF.to_csv('./_data/open/sample_submission_answer_ENBT_%s.csv' %ct, index=False)